In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- MODIFIABLE PARAMETERS ---
TARGET_BIS = 50.0
EPISODES = 5000
STEPS_PER_EP = 3600  # 10 minutes (5s steps)
ALPHA = 0.2         # Learning rate
GAMMA = 0.69        # Discount factor
EPSILON = 0.05      # Exploration

# --- SCHNIDER PK/PD CONSTANTS ---
V1, ke0 = 4.27, 0.17
k10, k12, k21, k13, k31 = 0.38, 0.30, 0.20, 0.19, 0.0035
BIS_0, BIS_MAX, EC50, HILL = 95.0, 75.0, 3.5, 2.5
ACTIONS = [0.0, 0.2, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0]

# --- ARTIFACT PERSISTENCE ---
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
Q_PATH = ARTIFACTS_DIR / "q_bis_deltabis_agent.npz"
TRACE_PATH = ARTIFACTS_DIR / "q_bis_deltabis_last_episode.npz"

BINS_PER_FEAT = 10
NUM_STATES = BINS_PER_FEAT**2  # 100 states (BIS error x deltaBIS, each with 10 bins)

def get_fuzzy_features(bis_error, deltabis):
    """Fuzzification using 2 membership functions: negative and positive

    Args:
        bis_error: BIS - TARGET (can be negative or positive)
        deltabis: BIS(t+1) - BIS(t-1)

    Returns:
        [mu_error_neg, mu_error_pos, mu_delta_neg, mu_delta_pos]
    """
    # For BIS error: scale by 20 (typical error range ±40)
    error_scaled = np.clip(bis_error / 20.0, -1, 1)
    mu_error_neg = max(0, -error_scaled)  # Negative membership
    mu_error_pos = max(0, error_scaled)   # Positive membership

    # For deltaBIS: scale by 20 (typical range ±40)
    delta_scaled = np.clip(deltabis / 20.0, -1, 1)
    mu_delta_neg = max(0, -delta_scaled)  # Negative membership
    mu_delta_pos = max(0, delta_scaled)   # Positive membership

    return np.array([mu_error_neg, mu_error_pos, mu_delta_neg, mu_delta_pos])

def get_state(bis_error, deltabis):
    """Map BIS error and deltaBIS to discrete state index"""
    features = get_fuzzy_features(bis_error, deltabis)

    # Discretize each of the 2 main features (error and delta)
    error_feat = (features[1] - features[0]) * (BINS_PER_FEAT - 1)  # neg to pos
    delta_feat = (features[3] - features[2]) * (BINS_PER_FEAT - 1)  # neg to pos

    error_bin = int(np.clip(error_feat, 0, BINS_PER_FEAT - 1))
    delta_bin = int(np.clip(delta_feat, 0, BINS_PER_FEAT - 1))

    idx = error_bin * BINS_PER_FEAT + delta_bin
    return idx

# --- LOAD OR INITIALIZE Q TABLE ---
if Q_PATH.exists():
    q_data = np.load(Q_PATH)
    Q = q_data["Q"]
    print(f"Loaded Q table from {Q_PATH}")
else:
    Q = np.zeros((NUM_STATES, len(ACTIONS)))
    print("Initialized new Q table")

bis_history = []
action_history = []

print(f"Starting Q-Learning training for {EPISODES} episodes...")
for ep in range(EPISODES):
    c1, c2, c3, ce = 0.0, 0.0, 0.0, 0.0
    bis_prev = BIS_0
    bis_curr = BIS_0
    bis_prev_prev = BIS_0

    for t in range(STEPS_PER_EP):
        bis_error = bis_curr - TARGET_BIS
        deltabis = bis_curr - bis_prev_prev

        s = get_state(bis_error, deltabis)

        # Epsilon-greedy selection
        a_idx = np.random.randint(len(ACTIONS)) if np.random.rand() < EPSILON else np.argmax(Q[s])
        u = ACTIONS[a_idx]

        # Update PK/PD (Euler 5s step)
        dt = 5/60
        c1 += (u - (k10 + k12 + k13)*c1 + k21*c2 + k31*c3) / V1 * dt
        c2 += (k12*c1 - k21*c2) * dt
        c3 += (k13*c1 - k31*c3) * dt
        ce += ke0 * (c1 - ce) * dt

        bis = BIS_0 - BIS_MAX * (ce**HILL / (ce**HILL + EC50**HILL))

        next_bis_error = bis - TARGET_BIS
        next_deltabis = bis - bis_prev
        reward = -abs(next_bis_error) - 0.5 * abs(next_deltabis)

        # Q-Learning update
        s_next = get_state(next_bis_error, next_deltabis)
        Q[s, a_idx] += ALPHA * (reward + GAMMA * np.max(Q[s_next]) - Q[s, a_idx])

        if ep == EPISODES - 1: # Log only the last episode for plotting
            bis_history.append(bis)
            action_history.append(u)

        bis_prev_prev = bis_prev
        bis_prev = bis_curr
        bis_curr = bis

    if (ep + 1) % 500 == 0:
        print(f"  Episode {ep + 1}/{EPISODES} completed")

# --- SAVE TRAINED Q TABLE AND TRACES ---
np.savez_compressed(
    Q_PATH,
    Q=Q,
    actions=np.array(ACTIONS, dtype=float),
    target_bis=np.array([TARGET_BIS], dtype=float),
    alpha=np.array([ALPHA], dtype=float),
    gamma=np.array([GAMMA], dtype=float),
    epsilon=np.array([EPSILON], dtype=float),
)
np.savez_compressed(
    TRACE_PATH,
    bis_history=np.array(bis_history, dtype=float),
    action_history=np.array(action_history, dtype=float),
)
print(f"Saved Q table to {Q_PATH}")
print(f"Saved last episode traces to {TRACE_PATH}")

# --- PLOTTING ---
plt.figure(figsize=(12, 6))

# Convert step indices to time (minutes)
if len(bis_history) > 0:
    t_sec = np.arange(len(bis_history)) * 5            # seconds
    t_min = t_sec / 60.0                               # minutes
else:
    t_min = np.array([])

max_min = (STEPS_PER_EP * 5) / 60.0
approx_ticks = 10
tick_step = max(1, int(np.ceil(max_min / approx_ticks)))
xticks = np.arange(0, max_min + 1e-8, tick_step)

# Top: BIS levels
plt.subplot(2, 1, 1)
plt.plot(t_min, bis_history, label="Measured BIS", color="blue")
plt.axhline(TARGET_BIS, color="red", linestyle="--", label=f"Target ({TARGET_BIS:.0f})")
plt.ylabel("BIS Index")
plt.title("Q-Learning Anesthesia Control Performance (BIS + deltaBIS with 2 membership functions)")
plt.ylim(0, 100)
plt.xlim(0, max_min)
plt.xticks(xticks)
plt.legend()
plt.grid(True, alpha=0.3)

# Bottom: Infusion rates
plt.subplot(2, 1, 2)
plt.step(t_min, action_history, color="green", where="post")
plt.ylabel("Infusion (ml/min)")
plt.xlabel("Time (minutes)")
plt.xlim(0, max_min)
plt.yticks(ACTIONS)
plt.xticks(xticks)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# EVALUATION ON POPULATION DATASET

In [ ]:
import pandas as pd
import random

# 1. Data Loading and Preprocessing
def load_data(path: str):
    return pd.read_csv(path)

def preprocess_data(df: pd.DataFrame):
    df = df.copy()
    def parse_age(x):
        try:
            parts = str(x).strip().split(" ")
            if len(parts) < 2: return 50
            low = int(parts[1])
            if len(parts) > 3:
                if parts[3] == 'older':
                    high = 100
                else:
                    high = int(parts[3])
                return int(random.randrange(low, high))
            return int(low)
        except Exception:
            return 50

    if "AgeCategory" in df.columns:
        df.loc[:,"AgeCategory"] = df["AgeCategory"].apply(parse_age)
    return df

def schnider_model(age: int, weight: float, height: float, sex: str) -> dict:
    sex = str(sex).lower()
    if sex == "male":
        lbm = 1.10 * weight - 128 * (weight ** 2) / (height ** 2)
    else:
        lbm = 1.07 * weight - 148 * (weight ** 2) / (height ** 2)

    V1 = 4.27
    V2 = 18.9 - 0.391 * (age - 53)
    V3 = 238.0

    k10 = 0.443 + 0.0107 * (weight - 77) - 0.0159 * (lbm - 59) + 0.0062 * (height - 177)
    k12 = 0.302 - 0.0056 * (age - 53)
    k13 = 0.196
    k21 = (1.29 - 0.024 * (age - 53)) / V2
    k31 = 0.0035
    ke0 = 0.456

    A = np.array([
        [-(k10 + k12 + k13),  k21,  k31,   0   ],
        [        k12,         -k21,   0,    0   ],
        [        k13,          0,   -k31,   0   ],
        [        ke0,          0,    0,   -ke0  ]
    ])
    B = np.array([[1 / V1], [0], [0], [0]])
    return {"A": A, "B": B}

def generate_schnider_dataset(df: pd.DataFrame):
    params_list = []
    for _, row in df.iterrows():
        params = schnider_model(
            age=row["AgeCategory"],
            weight=row["WeightInKilograms"],
            height=row["HeightInMeters"],
            sex=row["Sex"]
        )
        params_list.append(params)

    params_df = pd.DataFrame(params_list)
    return pd.concat([df.reset_index(drop=True), params_df], axis=1)

# 2. Evaluator for Q-Learning Agent (100 states, 2 features: BIS error and deltaBIS)
class QEvaluator:
    def __init__(self, q_table, actions):
        self.Q = q_table
        self.actions = actions
        self.target = 50.0
        self.bis0 = 95.0
        self.bis_max = 75.0
        self.ec50 = 3.5
        self.hill = 2.5
        self.bins_per_feat = 10

    def _get_state_idx(self, bis_error, deltabis):
        """Match feature extraction from Q-learning part exactly"""
        # Create fuzzy features
        error_scaled = np.clip(bis_error / 20.0, -1, 1)
        mu_error_neg = max(0, -error_scaled)
        mu_error_pos = max(0, error_scaled)

        delta_scaled = np.clip(deltabis / 20.0, -1, 1)
        mu_delta_neg = max(0, -delta_scaled)
        mu_delta_pos = max(0, delta_scaled)

        features = np.array([mu_error_neg, mu_error_pos, mu_delta_neg, mu_delta_pos])

        # Discretize to state index
        error_feat = (features[1] - features[0]) * (self.bins_per_feat - 1)
        delta_feat = (features[3] - features[2]) * (self.bins_per_feat - 1)

        error_bin = int(np.clip(error_feat, 0, self.bins_per_feat - 1))
        delta_bin = int(np.clip(delta_feat, 0, self.bins_per_feat - 1))

        idx = error_bin * self.bins_per_feat + delta_bin
        idx = min(idx, len(self.Q) - 1)
        return idx

    def simulate(self, patient_row, duration_min=120):
        dt = 5/60
        steps = int(duration_min / dt)

        A = np.asarray(patient_row['A'], dtype=float)
        B = np.asarray(patient_row['B'], dtype=float)

        x = np.zeros((4, 1), dtype=float)

        pe_log = []
        bis_log = []
        bis_prev_prev = self.bis0
        bis_prev = self.bis0

        MAX_CONCENTRATION = 10.0

        for t in range(steps):
            ce = float(x[3, 0])
            ce = np.clip(ce, 0.0, MAX_CONCENTRATION)

            if not np.isfinite(ce):
                ce = 0.0
                x = np.zeros((4, 1), dtype=float)

            ce_h = np.power(ce, self.hill, dtype=np.float64)
            ce_h = np.clip(ce_h, 0, 1e6)

            ec50_h = np.power(self.ec50, self.hill, dtype=np.float64)
            ec50_h = np.clip(ec50_h, 0, 1e6)

            denom = ce_h + ec50_h
            if denom > 0 and np.isfinite(denom):
                bis_ideal = self.bis0 - self.bis_max * (ce_h / denom)
            else:
                bis_ideal = self.bis0

            if not np.isfinite(bis_ideal):
                bis_ideal = self.bis0

            bis_ideal = np.clip(float(bis_ideal), 0, 100)
            measured_bis = bis_ideal + np.random.normal(0, 3)
            measured_bis = np.clip(measured_bis, 0, 100)

            # Compute state from BIS error and deltaBIS
            bis_error = float(measured_bis - self.target)
            deltabis = float(measured_bis - bis_prev_prev) if t > 0 else 0.0

            bis_error = 0.0 if not np.isfinite(bis_error) else bis_error
            deltabis = 0.0 if not np.isfinite(deltabis) else deltabis

            bis_error = np.clip(bis_error, -50.0, 50.0)
            deltabis = np.clip(deltabis, -30.0, 30.0)

            s_idx = self._get_state_idx(bis_error, deltabis)

            # Greedy Action
            a_idx = int(np.argmax(self.Q[s_idx]))
            u = float(self.actions[a_idx])

            x_dot = A @ x + B * u
            x = x + x_dot * dt
            x = np.clip(x, -MAX_CONCENTRATION, MAX_CONCENTRATION)

            error = measured_bis - self.target
            pe_log.append(100.0 * error / self.target if self.target != 0 else 0.0)
            bis_log.append(float(measured_bis))

            bis_prev_prev = bis_prev
            bis_prev = measured_bis

        return bis_log, pe_log

    def evaluate(self, df):
        summary = []
        for idx, row in df.iterrows():
            try:
                bis, pe = self.simulate(row)
                mdpe = np.median(pe)
                mdape = np.median(np.abs(pe))
                wobble = np.median(np.abs(pe - mdpe))
                controlled = (np.abs(np.array(bis) - self.target) <= 5).mean() * 100
                summary.append({
                    'PatientID': row['PatientID'],
                    'MDPE': mdpe, 'MDAPE': mdape,
                    'Wobble': wobble, 'Controlled (%)': controlled
                })
            except Exception as e:
                print(f"Warning: Failed to evaluate patient {row['PatientID']}: {e}")
                continue
        return pd.DataFrame(summary)

# 3. Execution
EVAL_SAMPLE_SIZE = 50

try:
    print("Loading Population Data...")
    raw_data = load_data("./data/Patients Data.csv")
    cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
    df_clean = preprocess_data(raw_data[cols])

    if EVAL_SAMPLE_SIZE and len(df_clean) > EVAL_SAMPLE_SIZE:
        print(f"Sampling {EVAL_SAMPLE_SIZE} patients from {len(df_clean)} total.")
        df_clean = df_clean.sample(n=EVAL_SAMPLE_SIZE, random_state=42)

    df_sim = generate_schnider_dataset(df_clean)

    print("Loading Q-Learning Agent...")
    q_data = np.load(Q_PATH)
    loaded_Q = q_data["Q"]
    loaded_actions = q_data["actions"]

    print("Evaluating on Population...")
    evaluator = QEvaluator(loaded_Q, loaded_actions)
    results_df = evaluator.evaluate(df_sim)

    print("\n--- Evaluation Results Summary ---")
    print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

    best_pt = results_df.loc[results_df['MDAPE'].idxmin()]
    print(f"\nBest Patient: {best_pt.to_dict()}")

except Exception as e:
    print(f"Evaluation failed: {e}")


# LOAD AND EVALUATE SAVED Q-LEARNING AGENT

In [ ]:
print("=" * 60)
print("LOADING SAVED Q-LEARNING AGENT (BIS + deltaBIS with 2 membership functions)")
print("=" * 60)

if not Q_PATH.exists():
    print(f"ERROR: Agent file not found at {Q_PATH}")
    print("Please run the Q-Learning training cell first.")
else:
    try:
        print(f"\nLoading agent from: {Q_PATH}")
        agent_data = np.load(Q_PATH)

        print(f"  - Loaded Q table: shape {agent_data['Q'].shape}")
        print(f"  - Actions available: {agent_data['actions']}")
        print(f"  - Target BIS: {agent_data['target_bis']}")
        print(f"  - Learning rate (alpha): {agent_data['alpha']}")
        print(f"  - Discount factor (gamma): {agent_data['gamma']}")
        print(f"  - Exploration rate (epsilon): {agent_data['epsilon']}")

        saved_Q = agent_data["Q"]
        saved_actions = agent_data["actions"]

        evaluator = QEvaluator(saved_Q, saved_actions)

        print("\n" + "=" * 60)
        print("EVALUATING ON POPULATION SAMPLE")
        print("=" * 60)

        raw_data = load_data("./data/Patients Data.csv")
        cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
        df_clean = preprocess_data(raw_data[cols])

        sample_size = 100
        if len(df_clean) > sample_size:
            print(f"\nSampling {sample_size} patients from {len(df_clean)} total...")
            df_clean = df_clean.sample(n=sample_size, random_state=42)

        print(f"Generating Schnider parameters for {len(df_clean)} patients...")
        df_sim = generate_schnider_dataset(df_clean)

        print(f"\nRunning evaluation simulation (120 min per patient)...")
        results_df = evaluator.evaluate(df_sim)

        if len(results_df) > 0:
            print("\n" + "=" * 60)
            print("EVALUATION RESULTS")
            print("=" * 60)
            print(f"\nEvaluated {len(results_df)} patients successfully\n")
            print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

            best_idx = results_df['MDAPE'].idxmin()
            worst_idx = results_df['MDAPE'].idxmax()

            print(f"\n--- Best Controlled Patient (ID: {results_df.loc[best_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[best_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[best_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[best_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[best_idx, 'Controlled (%)']:.2f}%")

            print(f"\n--- Worst Controlled Patient (ID: {results_df.loc[worst_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[worst_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[worst_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[worst_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[worst_idx, 'Controlled (%)']:.2f}%")
        else:
            print("No patients were successfully evaluated.")

    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        traceback.print_exc()
